In [ ]:
import cdsapi

dataset = "reanalysis-era5-single-levels"
request = {
    "product_type": ["reanalysis"],
    "variable": [
        "10m_u_component_of_wind", # Angin arah Barat-Timur
        "10m_v_component_of_wind", # Angin arah Utara-Selatan
        "2m_temperature", # Suhu permukaan
        "surface_pressure", # Tekanan permukaan
    ],
    "year": ["2026"],
    "month": ["02"],
    "day": [
        "01", "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12",
        "13", "14", "15",
        "16", "17", "18",
        "19", "20", "21",
        "22", "23", "24",
        "25", "26", "27",
        "28"
    ],
    "time": [
        "00:00", "01:00", "02:00",
        "03:00", "04:00", "05:00",
        "06:00", "07:00", "08:00",
        "09:00", "10:00", "11:00",
        "12:00", "13:00", "14:00",
        "15:00", "16:00", "17:00",
        "18:00", "19:00", "20:00",
        "21:00", "22:00", "23:00"
    ],
    "data_format": "grib",
    "download_format": "unarchived",
    "area": [6, 95, -6, 106.5]
}

In [2]:
client = cdsapi.Client()
client.retrieve(dataset, request).download()

2026-03-12 18:06:55,051 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-03-12 18:06:55,053 INFO Request ID is 0d1856e1-eb0c-47b7-b1d2-ba10b166df8d
2026-03-12 18:06:55,331 INFO status has been updated to accepted
2026-03-12 18:07:09,957 INFO status has been updated to running
2026-03-12 18:09:51,685 INFO status has been updated to successful


124479f12092bded9f99b11869fbb3a9.grib:   0%|          | 0.00/12.1M [00:00<?, ?B/s]

'124479f12092bded9f99b11869fbb3a9.grib'

In [8]:
import xarray as xr

# 1. Ambil variabel yang "normal" (suhu, angin, tekanan)
ds_main = xr.open_dataset(
    "sumatera_single_level.grib", 
    engine="cfgrib",
    backend_kwargs={'filter_by_keys': {'typeOfLevel': 'surface'}}
)

# 2. Ambil variabel "tp" secara spesifik (yang tadi kena skip)
ds_tp = xr.open_dataset(
    "sumatera_single_level.grib", 
    engine="cfgrib",
    backend_kwargs={'filter_by_keys': {'shortName': 'tp'}}
)

# 3. Gabungkan keduanya jadi satu Dataset
ds_final = xr.merge([ds_main, ds_tp], compat='override')

print(ds_final)

<xarray.Dataset> Size: 25MB
Dimensions:     (time: 672, latitude: 49, longitude: 47)
Coordinates:
  * time        (time) datetime64[ns] 5kB 2026-02-01 ... 2026-02-28T23:00:00
    valid_time  (time) datetime64[ns] 5kB ...
  * latitude    (latitude) float64 392B 6.0 5.75 5.5 5.25 ... -5.5 -5.75 -6.0
  * longitude   (longitude) float64 376B 95.0 95.25 95.5 ... 106.0 106.2 106.5
    number      int64 8B ...
    step        timedelta64[ns] 8B ...
    surface     float64 8B ...
Data variables:
    u10         (time, latitude, longitude) float32 6MB ...
    v10         (time, latitude, longitude) float32 6MB ...
    t2m         (time, latitude, longitude) float32 6MB ...
    sp          (time, latitude, longitude) float32 6MB ...
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centr

In [9]:
df_subset = ds_final.sel().to_dataframe().reset_index()
df_subset.dropna(inplace=True)

In [10]:
df_subset

,time,latitude,longitude,number,step,surface,valid_time,u10,v10,t2m,sp
0,2026-02-01 00:00:00,6.0,95.00,0,0 days,0.0,2026-02-01 00:00:00,-5.624283,-2.132980,300.812988,101168.4375
1,2026-02-01 00:00:00,6.0,95.25,0,0 days,0.0,2026-02-01 00:00:00,-5.433853,-1.257980,300.861816,101186.4375
2,2026-02-01 00:00:00,6.0,95.50,0,0 days,0.0,2026-02-01 00:00:00,-4.668228,-0.316574,300.785645,101233.4375
3,2026-02-01 00:00:00,6.0,95.75,0,0 days,0.0,2026-02-01 00:00:00,-4.273697,-0.208176,300.787598,101297.4375
4,2026-02-01 00:00:00,6.0,96.00,0,0 days,0.0,2026-02-01 00:00:00,-4.218033,-0.741379,300.885254,101296.4375
...,...,...,...,...,...,...,...,...,...,...,...
1547611,2026-02-28 23:00:00,-6.0,105.50,0,0 days,0.0,2026-02-28 23:00:00,3.876678,-0.784698,301.075195,100828.6250
1547612,2026-02-28 23:00:00,-6.0,105.75,0,0 days,0.0,2026-02-28 23:00:00,3.367889,0.400848,300.936523,100424.6250
1547613,2026-02-28 23:00:00,-6.0,106.00,0,0 days,0.0,2026-02-28 23:00:00,1.286835,0.784637,299.362305,99891.6250
1547614,2026-02-28 23:00:00,-6.0,106.25,0,0 days,0.0,2026-02-28 23:00:00,-0.023712,0.557098,299.178711,100706.6250


In [ ]:
df_subset.to_csv("data/data_sumatera.csv", index=False)